### Problem statement

The data is related to a coding platform that hosts coding challenges. They have a unique business model, where they crowdsource problems from various creators(authors). These authors create the problem and release it on the client's platform. The users then select the challenges they want to solve. 

This dataset contains information about each coding problem. It contains information about the problem, about the author who created it and a list of users who have attempted the problem.

Below are the fields that can be founf within each document in the collection -

- `challenge_id` - Unique id of the challenge problem

- `programming_language` - Programming language for the challenge

- `total_submissions` - Total submissions by all users

- `publish_date` - Publishing date for the challenge

- `author` - Embedded document about the author of the challenge.
> - `id` - Author id
> - `gender` - Author gender
> - `org_id` - Organisation if for author

- `users` - List of users who have attempted the challenge

----

### Connecting to MongoDB


----

In [1]:
# Importing the required libraries
import pymongo
import pprint as pp


pp.sorted = lambda x, key=None: x

In [2]:
client = pymongo.MongoClient("mongodb://localhost:27017/")

---
### Importing data

----

In [3]:
# # Restore database
# !mongorestore /home/avadmin/Desktop/Mongo/Content/Querying/Assignment/Data/querying_assignment

In [4]:
db = client.sourav

In [5]:
db.list_collection_names()

['Assign', 'Assignment', 'test_1']

In [6]:
pp.pprint(
    db.Assign.find_one()
)

{'_id': '60dab9f75945974466d8d64f',
 'challenge_id': 'CI23480',
 'programming_language': 1,
 'total_submissions': 15,
 'publish_date': '2002-10-16T00:00:00',
 'author': {'id': 'AI563578', 'gender': 'M', 'org_id': 'AOI100003'},
 'users': [60045]}


---
### Assignment Questions

----

### Q1. 

Find the number of documents in the collection

In [7]:
count = db.Assign.estimated_document_count()
print(f"Total documents: {count}")

Total documents: 5606


### Q2. 

Find the number of unique `programming_language` and `challenge_id`

In [8]:
# Unique programming languages
print("Unique Programming Languages :", len(db.Assign.distinct("programming_language")))

# Unique challenge IDs
print("Unique Challenge IDs :", len(db.Assign.distinct("challenge_id")))

Unique Programming Languages : 3
Unique Challenge IDs : 5606


### Q3. 

How many documents are there where the challenge was created between `2009-01-01` and `2010-01-01`? 

In [9]:
from datetime import datetime

start_date = datetime(2009, 1, 1)
end_date = datetime(2010, 1, 1)

count = db.Assign.count_documents({
    "publish_date": {
        "$gte": start_date,
        "$lt": end_date
    }
})

print(f"Challenges published between 2009 and 2010 : {count}")

Challenges published between 2009 and 2010 : 0


### Q4. 

How many challenges have been written by author `AI563576` in either `programming_language` `1` or `3` ?


In [10]:
query = {
    "author.id": "AI563576",
    "programming_language": {"$in": [1, 3]}
}
count = db.Assign.count_documents(query)
print(f"Challenges by author AI563576 in language 1 or 3 : {count}")

Challenges by author AI563576 in language 1 or 3 : 41


### Q5. 

How many documents are there where the challenges have been created by a female author and where the author belong to either the 'AOI100013' organisation or the 'AOI100018' organisation?

In [11]:
query = {
    "author.gender": "female",
    "author.org_id": {"$in": ["AOI100013", "AOI100018"]}
}
count = db.Assign.count_documents(query)
print(f"Challenges by female authors from given orgs : {count}")

Challenges by female authors from given orgs : 0


### Q6.

Find the top 5 challenge ids where either the challenge has been attempted by exactly 100 `users` or where the `total_submissions` is between 100 and 200, both inclusive?

***Hint - Think of using the `$size` operator.***

In [12]:
query = {
    "$or": [
        {"users": {"$size": 100}},
        {"total_submissions": {"$gte": 100, "$lte": 200}}
    ]
}
result = db.Assign.find(query, {"challenge_id": 1, "_id": 0}).limit(5)
for doc in result:
    pp.pprint(doc)

{'challenge_id': 'CI23482'}
{'challenge_id': 'CI23494'}
{'challenge_id': 'CI23500'}
{'challenge_id': 'CI23497'}
{'challenge_id': 'CI23519'}


### Q7. 

How documents are there where either the `publish_date > 2010-01-01` and `total_submissions > 100`, or the `publish_date < 2000-01-01` and `total_submissions > 1000` ?

In [13]:
from datetime import datetime

query = {
    "$or": [
        {
            "publish_date": {"$gt": datetime(2010, 1, 1)},
            "total_submissions": {"$gt": 100}
        },
        {
            "publish_date": {"$lt": datetime(2000, 1, 1)},
            "total_submissions": {"$gt": 1000}
        }
    ]
}
count = db.Assign.count_documents(query)
print(f"Matching documents : {count}")

Matching documents : 0
